# NusaSiaga — Wildfire Intelligence Pipeline

**Gemma 4 Good Hackathon · Kaggle · 2024**

---

## Overview

NusaSiaga is an **offline-first environmental disaster intelligence system** for Indonesia.

This notebook implements the full wildfire hotspot processing pipeline:

1. **Ingest** — Load real NASA FIRMS CSV (with sample fallback)
2. **Clean** — Normalize FIRMS fields, handle malformed rows
3. **Score** — Deterministic risk scoring with FRP + confidence + proximity weighting
4. **Classify** — CRITICAL / HIGH / MEDIUM / LOW severity labeling
5. **Summarize** — Province/regency summaries + environmental impact estimates
6. **Export** — Structured JSON outputs for the Next.js dashboard
7. **Prompt** — Gemma reasoning prompt examples for local AI inference

### Why deterministic scoring?

In disaster response, **explainability matters more than model accuracy**. A field coordinator needs to understand *why* a hotspot is CRITICAL. Our deterministic scoring formula uses satellite-verified physical measurements (FRP, brightness, confidence) combined with environmental context — no black-box ML required.

### Data Source

- **NASA FIRMS** (Fire Information for Resource Management System)
- MODIS Collection 6.1 + VIIRS 375m active fire products
- Coverage: Indonesia — Sumatera, Kalimantan, Java focus regions
- Download at: https://firms.modaps.eosdis.nasa.gov/

---


## 1. Environment Setup

Standard imports only. No external ML dependencies required — this pipeline runs fully offline.

In [ ]:
import os
import csv
import json
import math
import statistics
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict

# Output paths
OUTPUT_DIR = Path('../outputs')
DATA_DIR = Path('../data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input CSV path (real FIRMS download)
FIRMS_CSV_PATH = DATA_DIR / 'firms_hotspots.csv'

print(f'NusaSiaga Pipeline v2.0.0')
print(f'Output directory: {OUTPUT_DIR.resolve()}')
print(f'FIRMS CSV path: {FIRMS_CSV_PATH.resolve()}')
print(f'FIRMS CSV exists: {FIRMS_CSV_PATH.exists()}')

## 2. NASA FIRMS Data — Sample Reference

If you don't have a real FIRMS CSV, this inline sample provides representative Indonesian hotspot data for pipeline development and Kaggle reproducibility.

**To use real data:** Download from https://firms.modaps.eosdis.nasa.gov/download/ → select Indonesia region → MODIS or VIIRS → save as `data/firms_hotspots.csv`

Real FIRMS columns include:
- `latitude`, `longitude` — WGS84 coordinates
- `brightness` — Brightness temperature (Kelvin) at 4μm channel
- `frp` — Fire Radiative Power (megawatts) — **key intensity metric**
- `confidence` — Detection confidence (0-100 for MODIS; low/nominal/high for VIIRS)
- `acq_date`, `acq_time` — Acquisition timestamp
- `satellite` — Terra, Aqua, or SNPP
- `version` — FIRMS product version

In [ ]:
# Inline sample data — used only when FIRMS CSV is unavailable
# Represents realistic active fire detections across Indonesia's main fire-prone regions

SAMPLE_FIRMS_DATA = [
    # Kalimantan Barat (West Borneo) — primary peatland fire zone
    {'latitude': -0.2873, 'longitude': 111.4753, 'brightness': 335.2, 'frp': 42.5,
     'confidence': 80, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -0.3112, 'longitude': 111.5021, 'brightness': 341.7, 'frp': 38.2,
     'confidence': 72, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -0.4521, 'longitude': 111.6234, 'brightness': 328.9, 'frp': 22.1,
     'confidence': 65, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -0.5043, 'longitude': 111.7892, 'brightness': 356.4, 'frp': 71.3,
     'confidence': 90, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -0.1987, 'longitude': 111.3210, 'brightness': 319.8, 'frp': 18.7,
     'confidence': 60, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    {'latitude': -0.6234, 'longitude': 111.8901, 'brightness': 365.1, 'frp': 89.4,
     'confidence': 91, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    # Kalimantan Tengah (Central Borneo) — peatland carbon zone
    {'latitude': -1.2341, 'longitude': 110.4532, 'brightness': 342.6, 'frp': 45.8,
     'confidence': 75, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -1.3890, 'longitude': 110.5671, 'brightness': 351.3, 'frp': 58.3,
     'confidence': 85, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -1.4521, 'longitude': 110.6234, 'brightness': 368.9, 'frp': 95.2,
     'confidence': 90, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    {'latitude': -1.5678, 'longitude': 110.7890, 'brightness': 378.2, 'frp': 112.7,
     'confidence': 92, 'acq_date': '2024-08-15', 'acq_time': '0115',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -1.6234, 'longitude': 110.8901, 'brightness': 362.1, 'frp': 67.8,
     'confidence': 78, 'acq_date': '2024-08-15', 'acq_time': '0430',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -1.1203, 'longitude': 110.3456, 'brightness': 325.4, 'frp': 25.6,
     'confidence': 72, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    # Kalimantan Tengah cluster 2
    {'latitude': -2.4532, 'longitude': 113.8901, 'brightness': 358.7, 'frp': 82.1,
     'confidence': 88, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -2.5671, 'longitude': 113.9234, 'brightness': 344.3, 'frp': 41.3,
     'confidence': 70, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -2.3456, 'longitude': 113.7654, 'brightness': 371.8, 'frp': 98.6,
     'confidence': 89, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    {'latitude': -2.6789, 'longitude': 114.0123, 'brightness': 339.5, 'frp': 35.7,
     'confidence': 68, 'acq_date': '2024-08-15', 'acq_time': '0115',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -2.1234, 'longitude': 113.5678, 'brightness': 355.2, 'frp': 54.2,
     'confidence': 82, 'acq_date': '2024-08-15', 'acq_time': '0430',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    # Kalimantan Selatan
    {'latitude': -3.2341, 'longitude': 114.5678, 'brightness': 348.9, 'frp': 48.9,
     'confidence': 74, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -3.4521, 'longitude': 114.7892, 'brightness': 361.4, 'frp': 66.4,
     'confidence': 74, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    # Riau (Sumatera eastern coast)
    {'latitude': -0.8901, 'longitude': 104.3456, 'brightness': 343.1, 'frp': 39.8,
     'confidence': 71, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -0.9234, 'longitude': 104.4567, 'brightness': 369.8, 'frp': 101.3,
     'confidence': 91, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    {'latitude': -1.0123, 'longitude': 104.5678, 'brightness': 332.4, 'frp': 28.4,
     'confidence': 63, 'acq_date': '2024-08-15', 'acq_time': '0115',
     'satellite': 'Terra', 'version': '6.1NRT'},
    # Sumatera Selatan (South Sumatra) — major peat fire corridor
    {'latitude': -2.7890, 'longitude': 104.8901, 'brightness': 376.3, 'frp': 108.9,
     'confidence': 91, 'acq_date': '2024-08-14', 'acq_time': '0130',
     'satellite': 'Terra', 'version': '6.1NRT'},
    {'latitude': -2.8901, 'longitude': 104.9234, 'brightness': 358.1, 'frp': 63.7,
     'confidence': 83, 'acq_date': '2024-08-14', 'acq_time': '0445',
     'satellite': 'Aqua', 'version': '6.1NRT'},
    {'latitude': -3.1234, 'longitude': 105.0123, 'brightness': 345.6, 'frp': 43.2,
     'confidence': 68, 'acq_date': '2024-08-14', 'acq_time': '0600',
     'satellite': 'SNPP', 'version': '2.0NRT'},
    {'latitude': -3.3456, 'longitude': 105.2345, 'brightness': 372.4, 'frp': 97.4,
     'confidence': 89, 'acq_date': '2024-08-15', 'acq_time': '0115',
     'satellite': 'Terra', 'version': '6.1NRT'},
]

print(f'Sample dataset: {len(SAMPLE_FIRMS_DATA)} hotspots')
print('Regions covered: Kalimantan Barat, Kalimantan Tengah, Kalimantan Selatan, Riau, Sumatera Selatan')

## 3. Province Lookup

Indonesia's fire-prone provinces are determined by bounding box classification. In production, this would use a proper administrative boundary shapefile (via GADM or BPS data). Here we use deterministic bounding boxes sufficient for disaster triage purposes.

In [ ]:
# Deterministic province/regency lookup by bounding box
# Covers the primary Indonesian fire-prone regions

PROVINCE_BOUNDS = [
    # (lat_min, lat_max, lon_min, lon_max, province, regency_hint)
    # Kalimantan Barat
    (-0.7, 0.0, 111.0, 112.5, 'Kalimantan Barat', 'Sintang / Melawi'),
    (-2.0, -0.7, 109.5, 111.5, 'Kalimantan Barat', 'Ketapang'),
    (-2.0, -1.0, 111.5, 113.0, 'Kalimantan Barat', 'Kapuas Hulu'),
    # Kalimantan Tengah
    (-2.0, -0.8, 110.0, 111.5, 'Kalimantan Tengah', 'Barito Selatan'),
    (-3.5, -2.0, 113.0, 115.0, 'Kalimantan Tengah', 'Kapuas / Pulang Pisau'),
    # Kalimantan Selatan
    (-4.0, -2.8, 114.0, 116.0, 'Kalimantan Selatan', 'Hulu Sungai Selatan'),
    # Riau (eastern Sumatera)
    (-1.5, 0.5, 103.5, 105.5, 'Riau', 'Pelalawan / Indragiri'),
    # Sumatera Selatan
    (-4.0, -2.5, 104.5, 106.5, 'Sumatera Selatan', 'OKI / Musi Banyuasin'),
]

# Known settlement coordinates (lat, lon, name) — for proximity scoring
SETTLEMENTS = [
    (-0.0251, 109.3425, 'Pontianak'),
    (-1.6831, 102.3614, 'Jambi'),
    (-2.9909, 104.7566, 'Palembang'),
    (0.5071,  101.4478, 'Pekanbaru'),
    (-1.4833, 112.9500, 'Palangkaraya'),
    (-3.3194, 114.5908, 'Banjarmasin'),
]

def lookup_province(lat: float, lon: float) -> tuple[str, str]:
    """Return (province, regency_hint) for given coordinates."""
    for lat_min, lat_max, lon_min, lon_max, province, regency in PROVINCE_BOUNDS:
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return province, regency
    return 'Indonesia (unclassified)', 'Unknown regency'

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate great-circle distance in kilometers."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def nearest_settlement_km(lat: float, lon: float) -> float:
    """Return distance (km) to nearest known settlement."""
    distances = [
        haversine_km(lat, lon, s[0], s[1])
        for s in SETTLEMENTS
    ]
    return min(distances)

print('Province lookup and settlement proximity functions ready.')

## 4. CSV Ingestion with Robust Normalization

Handles both real NASA FIRMS CSV and the inline sample fallback.

FIRMS CSVs from different satellites (MODIS vs VIIRS) have slightly different schemas:
- MODIS: `confidence` is numeric (0–100)
- VIIRS: `confidence` is categorical (`low`, `nominal`, `high`)

We normalize both to a 0–100 numeric scale.

In [ ]:
def normalize_confidence(raw: str) -> int:
    """
    Normalize FIRMS confidence to 0-100 integer.
    MODIS: already numeric string (e.g. '80')
    VIIRS: categorical ('low' -> 40, 'nominal' -> 65, 'high' -> 90)
    """
    raw = str(raw).strip().lower()
    viirs_map = {'low': 40, 'nominal': 65, 'high': 90}
    if raw in viirs_map:
        return viirs_map[raw]
    try:
        val = int(float(raw))
        return max(0, min(100, val))
    except (ValueError, TypeError):
        return 60  # safe default

def parse_firms_row(row: dict) -> dict | None:
    """
    Parse and normalize a FIRMS CSV row (or dict from sample data).
    Returns None for malformed/unreadable rows.
    """
    try:
        lat = float(row.get('latitude', row.get('lat', '')))
        lon = float(row.get('longitude', row.get('lon', '')))
        
        # Validate coordinates are within Indonesia bounds
        if not (-12.0 <= lat <= 8.0 and 94.0 <= lon <= 142.0):
            return None
        
        brightness_raw = row.get('brightness', row.get('bright_ti4', 300))
        brightness = float(brightness_raw) if brightness_raw else 300.0
        
        frp_raw = row.get('frp', 0)
        frp = float(frp_raw) if frp_raw else 0.0
        
        confidence_raw = row.get('confidence', '60')
        confidence = normalize_confidence(confidence_raw)
        
        return {
            'latitude': round(lat, 4),
            'longitude': round(lon, 4),
            'brightness': round(brightness, 1),
            'frp': round(frp, 1),
            'confidence': confidence,
            'acq_date': str(row.get('acq_date', 'unknown')).strip(),
            'acq_time': str(row.get('acq_time', '')).strip(),
            'satellite': str(row.get('satellite', 'unknown')).strip(),
            'version': str(row.get('version', '')).strip(),
        }
    except (ValueError, TypeError, KeyError):
        return None  # skip malformed rows silently

def load_firms_data() -> tuple[list[dict], str]:
    """
    Load FIRMS data. Returns (rows, source_label).
    Prefers real CSV; falls back to inline sample.
    """
    if FIRMS_CSV_PATH.exists():
        print(f'Loading real FIRMS CSV: {FIRMS_CSV_PATH}')
        rows = []
        with open(FIRMS_CSV_PATH, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for raw_row in reader:
                parsed = parse_firms_row(raw_row)
                if parsed:
                    rows.append(parsed)
        print(f'  Loaded {len(rows)} valid rows from CSV')
        return rows, 'NASA FIRMS CSV (real data)'
    else:
        print('FIRMS CSV not found — using inline sample data')
        rows = []
        for item in SAMPLE_FIRMS_DATA:
            parsed = parse_firms_row(item)
            if parsed:
                rows.append(parsed)
        print(f'  Loaded {len(rows)} sample rows')
        return rows, 'Inline sample data (demo)'

firms_rows, data_source_label = load_firms_data()
print(f'\nData source: {data_source_label}')
print(f'Total valid hotspots: {len(firms_rows)}')

## 5. Risk Scoring — Deterministic FRP-Weighted Model

### Scoring Formula

Each hotspot receives a **risk score from 0–100** based on four components:

| Component | Weight | Rationale |
|---|---|---|
| FRP score | 40% | Fire Radiative Power is the strongest physical indicator of fire intensity |
| Brightness score | 25% | Temperature at 4μm channel indicates heat output |
| Confidence score | 20% | Detection certainty; low confidence = lower base risk |
| Settlement proximity | 15% | Fires near populated areas require faster response |

### Severity Classification

| Score | Severity | Response Level |
|---|---|---|
| 80–100 | CRITICAL | Immediate response — possible evacuation |
| 60–79 | HIGH | Priority monitoring — alert responders |
| 40–59 | MEDIUM | Active monitoring — 6-hour review cycle |
| 0–39 | LOW | Watchlist — 24-hour review cycle |

### Environmental Labels

Hotspots within known peatland zones receive additional environmental labeling because peat fires:
- Release 10–40× more carbon per hectare than forest fires
- Produce toxic PM2.5-heavy smoke
- Can smolder underground for weeks, resistant to conventional firefighting

In [ ]:
# Known peatland zones in Indonesia (simplified bounding boxes)
# Source: Global Peatland Database / Wetlands International approximations
PEATLAND_ZONES = [
    # (lat_min, lat_max, lon_min, lon_max, zone_name)
    (-2.5, 0.5, 108.0, 114.0, 'Kalimantan peatland belt'),
    (-4.5, -1.5, 103.0, 107.0, 'Sumatera peatland zone'),
    (-1.5, 1.5, 100.0, 104.0, 'Riau peatland corridor'),
]

def is_peatland(lat: float, lon: float) -> bool:
    """Check if coordinate falls within a known peatland zone."""
    for lat_min, lat_max, lon_min, lon_max, _ in PEATLAND_ZONES:
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return True
    return False

def score_frp(frp: float) -> float:
    """Normalize FRP to 0-100. 100 MW FRP = score 80; 200+ MW = score 100."""
    if frp <= 0:
        return 10.0
    # Logarithmic scale — reflects physical fire intensity
    score = min(100.0, (math.log10(max(frp, 1)) / math.log10(200)) * 100)
    return round(score, 1)

def score_brightness(brightness: float) -> float:
    """Normalize brightness temperature to 0-100. 310K = baseline; 400K+ = max."""
    baseline = 300.0
    ceiling = 400.0
    score = max(0.0, min(100.0, (brightness - baseline) / (ceiling - baseline) * 100))
    return round(score, 1)

def score_proximity(lat: float, lon: float) -> float:
    """Settlement proximity score. Closer = higher urgency = higher score."""
    dist_km = nearest_settlement_km(lat, lon)
    # Within 25km = 100; 200km+ = 0
    score = max(0.0, min(100.0, (1 - dist_km / 200.0) * 100))
    return round(score, 1)

def compute_risk_score(row: dict) -> int:
    """
    Compute deterministic risk score (0-100).
    Weights: FRP 40%, brightness 25%, confidence 20%, proximity 15%.
    """
    frp_s     = score_frp(row['frp'])
    bright_s  = score_brightness(row['brightness'])
    conf_s    = float(row['confidence'])  # already 0-100
    prox_s    = score_proximity(row['latitude'], row['longitude'])
    
    score = (frp_s * 0.40 + bright_s * 0.25 + conf_s * 0.20 + prox_s * 0.15)
    return round(score)

def classify_severity(score: int) -> str:
    if score >= 80: return 'CRITICAL'
    if score >= 60: return 'HIGH'
    if score >= 40: return 'MEDIUM'
    return 'LOW'

def build_environmental_label(row: dict, score: int, peat: bool) -> str:
    """Generate concise environmental label for dashboard display."""
    if peat and score >= 80:
        return 'Peatland fire — extreme carbon release risk'
    if peat and score >= 60:
        return 'Active peatland fire — monitoring required'
    if peat:
        return 'Low FRP — possible smoldering peat'
    if score >= 80:
        return 'High-intensity fire — settlement proximity risk'
    if row['frp'] < 25:
        return 'Low FRP — possible smoldering vegetation'
    return 'Active burn zone — standard monitoring'

def estimate_smoke_impact(score: int, peat: bool) -> str:
    if score >= 80 or (peat and score >= 60): return 'severe'
    if score >= 65: return 'high'
    if score >= 50: return 'moderate'
    if score >= 35: return 'low'
    return 'minimal'

print('Risk scoring functions defined.')

## 6. Pipeline Execution — Score and Classify All Hotspots

In [ ]:
scored_hotspots = []

for idx, row in enumerate(firms_rows):
    lat = row['latitude']
    lon = row['longitude']
    
    province, regency = lookup_province(lat, lon)
    peat = is_peatland(lat, lon)
    risk_score = compute_risk_score(row)
    severity = classify_severity(risk_score)
    env_label = build_environmental_label(row, risk_score, peat)
    smoke = estimate_smoke_impact(risk_score, peat)
    
    scored_hotspots.append({
        'id': f'HS-{idx+1:03d}',
        'lat': lat,
        'lon': lon,
        'severity': severity,
        'risk_score': risk_score,
        'brightness': row['brightness'],
        'frp': row['frp'],
        'confidence': row['confidence'],
        'satellite': row['satellite'],
        'acq_date': row['acq_date'],
        'province': province,
        'regency': regency,
        'peatland': peat,
        'environmental_label': env_label,
        'smoke_impact': smoke,
    })

# Sort by risk_score descending
scored_hotspots.sort(key=lambda h: h['risk_score'], reverse=True)

# Print summary
severity_counts = {'CRITICAL': 0, 'HIGH': 0, 'MEDIUM': 0, 'LOW': 0}
for h in scored_hotspots:
    severity_counts[h['severity']] += 1

print('=== Pipeline Results ===')
print(f'Total hotspots scored: {len(scored_hotspots)}')
print(f'CRITICAL: {severity_counts["CRITICAL"]}')
print(f'HIGH:     {severity_counts["HIGH"]}')
print(f'MEDIUM:   {severity_counts["MEDIUM"]}')
print(f'LOW:      {severity_counts["LOW"]}')

frps = [h['frp'] for h in scored_hotspots if h['frp'] > 0]
if frps:
    print(f'Max FRP:  {max(frps):.1f} MW')
    print(f'Avg FRP:  {statistics.mean(frps):.1f} MW')

print('\nTop 5 hotspots by risk score:')
for h in scored_hotspots[:5]:
    print(f"  {h['id']} | {h['severity']:8s} | score={h['risk_score']:3d} | frp={h['frp']:6.1f} MW | {h['province']}")

## 7. Province Summaries

Aggregate hotspot statistics by province for the dashboard summary panel.

In [ ]:
province_groups: dict[str, list[dict]] = defaultdict(list)
for h in scored_hotspots:
    province_groups[h['province']].append(h)

province_summaries = []
for province, hotspots in sorted(province_groups.items()):
    critical = sum(1 for h in hotspots if h['severity'] == 'CRITICAL')
    high = sum(1 for h in hotspots if h['severity'] == 'HIGH')
    frps_p = [h['frp'] for h in hotspots if h['frp'] > 0]
    
    risk_level = 'CRITICAL' if critical > 0 else ('HIGH' if high > 0 else 'MEDIUM')
    regencies = list(set(h['regency'] for h in hotspots if h['regency']))
    
    summary = {
        'province': province,
        'hotspot_count': len(hotspots),
        'critical_count': critical,
        'avg_frp': round(statistics.mean(frps_p), 1) if frps_p else 0,
        'max_frp': round(max(frps_p), 1) if frps_p else 0,
        'risk_level': risk_level,
        'regencies_affected': regencies[:5],  # top 5
    }
    province_summaries.append(summary)

# Sort by critical count then hotspot count
province_summaries.sort(key=lambda p: (-p['critical_count'], -p['hotspot_count']))

print('Province summaries:')
for ps in province_summaries:
    print(f"  {ps['province']:35s} | {ps['hotspot_count']:2d} hotspots | {ps['critical_count']} CRITICAL | max FRP {ps['max_frp']:.1f} MW")

## 8. Environmental Impact Estimation

Deterministic estimates for environmental impact summary. These are heuristic estimates based on established fire science parameters:

- **Area burned**: ~10 ha per MW of total FRP (rough empirical conversion)
- **Carbon release**: Peatland fires release ~750 tons CO₂-eq per hectare (IPCC estimates)
- **Smoke plume radius**: Based on total FRP and peatland involvement

In [ ]:
total_frp = sum(h['frp'] for h in scored_hotspots)
peat_hotspots = [h for h in scored_hotspots if h['peatland']]
peat_fraction = len(peat_hotspots) / max(len(scored_hotspots), 1)

# Heuristic area estimate: ~5-15 ha per MW of FRP
estimated_area_ha = round(total_frp * 8)

# Carbon estimate: peat=~750 tons/ha, forest=~180 tons/ha
peat_area = estimated_area_ha * peat_fraction
forest_area = estimated_area_ha * (1 - peat_fraction)
estimated_co2 = round(peat_area * 750 + forest_area * 180)

# Smoke plume radius: based on FRP magnitude
smoke_radius_km = min(200, round(math.sqrt(total_frp) * 4))

# AQI risk level
critical_count = severity_counts['CRITICAL']
if critical_count >= 4:
    aqi_risk = 'HAZARDOUS'
elif critical_count >= 2 or peat_fraction > 0.5:
    aqi_risk = 'VERY_UNHEALTHY'
elif critical_count >= 1:
    aqi_risk = 'UNHEALTHY'
else:
    aqi_risk = 'UNHEALTHY_SENSITIVE_GROUPS'

# Evacuation recommendation
if critical_count >= 3:
    evac_rec = (f'Recommend evacuation advisory for settlements within '
                f'{min(50, smoke_radius_km//2)}km of CRITICAL hotspot clusters. '
                f'Priority provinces: {", ".join(p["province"] for p in province_summaries[:2] if p["critical_count"] > 0)}.')
elif critical_count >= 1:
    evac_rec = 'Issue protective action advisories near CRITICAL hotspots. Monitor for spread.'
else:
    evac_rec = 'No immediate evacuation required. Maintain active monitoring.'

smoke_hazard = (f'Active wildfire cluster generating smoke plume across \'~{smoke_radius_km}km radius. '
                 f'PM2.5 levels likely elevated in downwind corridors. '
                 f'AQI risk category: {aqi_risk}. '
                 f'Peatland involvement: {"YES" if peat_hotspots else "MINIMAL"}.')

environmental_impact = {
    'estimated_area_burned_ha': estimated_area_ha,
    'smoke_plume_radius_km': smoke_radius_km,
    'aqi_risk_level': aqi_risk,
    'estimated_co2_release_tons': estimated_co2,
    'peatland_involvement': len(peat_hotspots) > 0,
    'smoke_hazard_estimate': smoke_hazard,
    'evacuation_recommendation': evac_rec,
    'carbon_release_note': (
        'Peatland fires contribute disproportionately high carbon per hectare. '
        'Long-duration smoldering may continue underground after visible fire suppression.'
        if peat_hotspots else
        'No significant peatland involvement detected in current scan.'
    )
}

print('Environmental Impact Estimates:')
print(f'  Estimated area burned: {estimated_area_ha:,} ha')
print(f'  Estimated CO₂ release: {estimated_co2:,} tons')
print(f'  Smoke plume radius: ~{smoke_radius_km} km')
print(f'  AQI risk level: {aqi_risk}')
print(f'  Peatland hotspots: {len(peat_hotspots)}/{len(scored_hotspots)}')

## 9. Export — Dashboard JSON

This is the primary output consumed by the NusaSiaga Next.js dashboard. The schema is stable — the frontend `dashboard-hotspots.ts` loader reads from this file.

In [ ]:
# Build summary block
frps_all = [h['frp'] for h in scored_hotspots if h['frp'] > 0]
provinces_list = list(dict.fromkeys(h['province'] for h in scored_hotspots
                                    if 'unclassified' not in h['province']))
satellites_used = list(set(h['satellite'] for h in scored_hotspots if h['satellite']))

dashboard_summary = {
    'total': len(scored_hotspots),
    'critical': severity_counts['CRITICAL'],
    'high': severity_counts['HIGH'],
    'medium': severity_counts['MEDIUM'],
    'low': severity_counts['LOW'],
    'avg_frp': round(statistics.mean(frps_all), 1) if frps_all else 0,
    'max_frp': round(max(frps_all), 1) if frps_all else 0,
    'satellites_used': sorted(satellites_used),
    'provinces_affected': provinces_list[:8],
}

# Strip internal-only field 'peatland' from dashboard export
dashboard_hotspots = [
    {k: v for k, v in h.items() if k != 'peatland'}
    for h in scored_hotspots
]

# Date range
dates = sorted(set(h['acq_date'] for h in scored_hotspots if h['acq_date'] != 'unknown'))

dashboard_output = {
    'metadata': {
        'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'source': 'NASA FIRMS MODIS/VIIRS via NusaSiaga Pipeline v2',
        'pipeline_version': '2.0.0',
        'data_source': data_source_label,
        'total_hotspots': len(scored_hotspots),
        'coverage_region': 'Indonesia',
        'date_range': {
            'start': dates[0] if dates else 'unknown',
            'end': dates[-1] if dates else 'unknown',
        },
        'scoring_model': 'deterministic_frp_weighted_v2',
    },
    'summary': dashboard_summary,
    'environmental_impact': environmental_impact,
    'province_summaries': province_summaries,
    'hotspots': dashboard_hotspots,
}

output_path = OUTPUT_DIR / 'dashboard_hotspots.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(dashboard_output, f, indent=2, ensure_ascii=False)

print(f'Dashboard JSON written: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024:.1f} KB')

## 10. Export — Gemma Prompt Examples

Generate structured reasoning prompts for local Gemma inference.
These demonstrate the NusaSiaga offline AI workflow.

In [ ]:
def build_hotspot_prompt(h: dict) -> dict:
    """Build a Gemma reasoning prompt for a single hotspot."""
    prompt = (
        f"You are NusaSiaga, an environmental disaster intelligence assistant for Indonesia.\n"
        f"Analyze this wildfire hotspot detected by NASA FIRMS satellites.\n\n"
        f"Hotspot Data:\n"
        f"- Location: {h['lat']:.4f}°{'S' if h['lat'] < 0 else 'N'}, {h['lon']:.4f}°E "
        f"({h['regency']}, {h['province']})\n"
        f"- Severity: {h['severity']}\n"
        f"- Risk Score: {h['risk_score']}/100\n"
        f"- Fire Radiative Power (FRP): {h['frp']:.1f} MW\n"
        f"- Brightness Temperature: {h['brightness']:.1f} K\n"
        f"- Detection Confidence: {h['confidence']}%\n"
        f"- Satellite: {h['satellite']}\n"
        f"- Date: {h['acq_date']}\n"
        f"- Environmental Label: {h['environmental_label']}\n"
        f"- Smoke Impact: {h['smoke_impact'].upper()}\n\n"
        f"Provide:\n"
        f"1. Hazard assessment for this hotspot\n"
        f"2. Estimated smoke and AQI impact on nearby communities\n"
        f"3. Key environmental risks\n"
        f"4. Evacuation or protective action recommendations\n"
        f"5. Priority for emergency response teams\n\n"
        f"Keep the analysis practical for field responders. Use clear, direct language."
    )
    return {
        'id': f"PROMPT-{h['id']}",
        'hotspot_id': h['id'],
        'context': f"{h['severity']} severity hotspot — {h['province']}",
        'prompt': prompt,
        'severity': h['severity'],
        'frp': h['frp'],
        'expected_reasoning_focus': [
            'fire_intensity_assessment',
            'smoke_health_impact',
            'environmental_risk',
            'response_prioritization',
        ]
    }

# Generate prompts for top CRITICAL and HIGH hotspots
prompt_hotspots = (
    [h for h in scored_hotspots if h['severity'] == 'CRITICAL'][:3] +
    [h for h in scored_hotspots if h['severity'] == 'HIGH'][:2]
)

# Cluster summary prompt
provinces_affected_str = ', '.join(provinces_list[:4])
cluster_prompt = {
    'id': 'PROMPT-CLUSTER-SUMMARY',
    'hotspot_id': 'cluster_summary',
    'context': f'Multi-province cluster summary — {len(scored_hotspots)} hotspots',
    'prompt': (
        f"You are NusaSiaga. Synthesize this multi-province wildfire cluster status.\n\n"
        f"Active Cluster Summary:\n"
        f"- Total hotspots: {len(scored_hotspots)}\n"
        f"- CRITICAL: {severity_counts['CRITICAL']} | HIGH: {severity_counts['HIGH']}\n"
        f"- Provinces: {provinces_affected_str}\n"
        f"- Max FRP: {max(frps_all):.1f} MW\n"
        f"- Estimated area: {estimated_area_ha:,} ha\n"
        f"- Estimated CO\u2082 release: {estimated_co2:,} tons\n"
        f"- AQI risk level: {aqi_risk}\n"
        f"- Peatland involvement: {'YES' if peat_hotspots else 'MINIMAL'}\n\n"
        f"Provide a structured intelligence brief for emergency coordinators covering:\n"
        f"1. Overall situation assessment\n"
        f"2. Population exposure and health risk\n"
        f"3. Carbon and climate significance\n"
        f"4. Provincial priority ranking\n"
        f"5. 24-48 hour risk trajectory"
    ),
    'severity': 'CRITICAL',
    'frp': None,
    'expected_reasoning_focus': ['cluster_synthesis', 'population_health', 'carbon_significance']
}

# AQI integration placeholder
aqi_placeholder = {
    'id': 'AQI-PROMPT-001',
    'status': 'PLACEHOLDER — requires OpenAQ data feed',
    'context': 'AQI-correlated smoke hazard prompt — ready for live data injection',
    'prompt_template': (
        "You are NusaSiaga. A wildfire cluster is producing smoke across Kalimantan. "
        "Current AQI at {station_name}: {aqi_value} ({aqi_category}). "
        "PM2.5: {pm25} μg/m³. Wind: {wind_dir} at {wind_speed} km/h.\n\n"
        "Assess: health risk for {population_count} residents, outdoor advisories, "
        "vulnerable population guidance, and AQI trajectory over 12 hours."
    ),
    'fields_required': ['station_name', 'aqi_value', 'aqi_category', 'pm25', 'wind_dir', 'wind_speed', 'population_count'],
    'openaq_endpoint': 'https://api.openaq.org/v3/locations?country=ID&parameter=pm25',
    'integration_notes': (
        'Connect to OpenAQ v3 API for live PM2.5 and AQI readings. '
        'Filter for Indonesian stations near active hotspot clusters. '
        'Inject live values into prompt_template fields before sending to local Gemma model.'
    )
}

gemma_output = {
    'metadata': {
        'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'pipeline_version': '2.0.0',
        'model_target': 'gemma3n:e2b',
        'prompt_style': 'structured_environmental_reasoning',
    },
    'prompts': [build_hotspot_prompt(h) for h in prompt_hotspots] + [cluster_prompt],
    'aqi_placeholder_prompts': [aqi_placeholder],
}

gemma_path = OUTPUT_DIR / 'gemma_prompt_examples.json'
with open(gemma_path, 'w', encoding='utf-8') as f:
    json.dump(gemma_output, f, indent=2, ensure_ascii=False)

print(f'Gemma prompts written: {gemma_path}')
print(f'Hotspot prompts: {len(prompt_hotspots)} | Cluster summary: 1 | AQI placeholder: 1')

## 11. Validation — Verify Output Schema

Confirm the generated JSON is valid and contains the minimum required fields for the dashboard.

In [ ]:
# Reload and validate dashboard_hotspots.json
with open(OUTPUT_DIR / 'dashboard_hotspots.json', 'r') as f:
    loaded = json.load(f)

assert 'hotspots' in loaded, 'Missing hotspots array'
assert 'summary' in loaded, 'Missing summary'
assert 'metadata' in loaded, 'Missing metadata'
assert 'environmental_impact' in loaded, 'Missing environmental_impact'
assert len(loaded['hotspots']) > 0, 'Hotspots array is empty'

required_hotspot_fields = ['id', 'lat', 'lon', 'severity', 'risk_score']
for i, h in enumerate(loaded['hotspots']):
    for field in required_hotspot_fields:
        assert field in h, f'Hotspot {i} missing required field: {field}'
    assert h['severity'] in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW'], \
        f"Hotspot {i} has invalid severity: {h['severity']}"
    assert 0 <= h['risk_score'] <= 100, \
        f"Hotspot {i} has out-of-range risk_score: {h['risk_score']}"

print('✓ dashboard_hotspots.json — schema validation passed')
print(f'  Hotspots: {len(loaded["hotspots"])}')
print(f'  Schema version compatible with frontend dashboard-hotspots.ts loader')

# Validate gemma_prompt_examples.json
with open(OUTPUT_DIR / 'gemma_prompt_examples.json', 'r') as f:
    gemma_loaded = json.load(f)

assert 'prompts' in gemma_loaded, 'Missing prompts array'
assert len(gemma_loaded['prompts']) > 0, 'Prompts array is empty'
print('✓ gemma_prompt_examples.json — validation passed')
print(f'  Prompts: {len(gemma_loaded["prompts"])}')
print()
print('Pipeline complete. All outputs ready for NusaSiaga dashboard.')

## 12. Summary — Future Integration Notes

### Using Real NASA FIRMS Data

1. Go to https://firms.modaps.eosdis.nasa.gov/download/
2. Select country: Indonesia
3. Select product: MODIS Collection 6.1 or VIIRS 375m
4. Select date range (recent 7 days recommended)
5. Download as CSV
6. Place file at `data/firms_hotspots.csv`
7. Re-run this notebook → outputs are automatically refreshed

### AQI Integration (Future)

The `aqi_placeholder_prompts` block in `gemma_prompt_examples.json` describes the planned OpenAQ v3 integration:
- Endpoint: `https://api.openaq.org/v3/locations?country=ID&parameter=pm25`
- Inject live PM2.5 readings into Gemma prompts for smoke health impact reasoning
- No API key required for basic public data

### Gemma Local Inference

This pipeline generates structured prompts for local Gemma reasoning via Ollama:
```bash
ollama pull gemma3n:e2b
ollama serve
```

The Next.js dashboard API route `/api/analyze` forwards prompts to `http://localhost:11434/api/generate`.

### Pipeline Architecture

```
NASA FIRMS CSV
      ↓
nusasiaga_gemma_pipeline.ipynb
      ↓
outputs/dashboard_hotspots.json     ← consumed by dashboard
outputs/gemma_prompt_examples.json  ← used for Gemma prompting
      ↓
src/lib/dashboard-hotspots.ts
      ↓
Next.js Dashboard (localhost:3000)
```